# Milestone 3 — Reference Matching Pipeline & Full Benchmarking

Notebook thực thi toàn bộ quy trình **Milestone 3** theo đặc tả [matching.md](../Project_description/matching.md) và [benchmark_m3.md](../Project_description/benchmark_m3.md):
1. **Kết nối Google Drive & Sao chép dữ liệu Parquet về SSD cục bộ** (`/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data/parquet/` $\rightarrow$ `/content/local_parquet/`).
2. **Data Cleaning & Labelling:** Gán nhãn thủ công ($\ge 5$ bài báo, mỗi bài $\ge 20$ mục) và gán nhãn tự động ($\ge 10\%$ số bài ngoài tập thủ công).
3. **Feature Engineering:** Xây dựng 16 đặc trưng cặp `(bibtex_entry, reference)` (bao gồm ngữ cảnh trích dẫn từ cây `hierarchy` của Milestone 2).
4. **Huấn luyện & Benchmark 4 Mô hình:**
   - Baseline 1: **TF-IDF Cosine thuần**
   - Baseline 2: **Logistic Regression**
   - Model 3: **Random Forest**
   - Model 4: **LightGBM**
5. **Đo Benchmark Tài nguyên Hệ thống (`m3_system_benchmark.csv`):** Đo thời gian từng pha, Peak/Avg RAM, Peak Disk và dung lượng thành phẩm bằng `SystemBenchmarkTracker`.
6. **Xuất `predictions.parquet`** từ mô hình tốt nhất (theo `Valid MRR@5`) và đồng bộ kết quả về Google Drive.


In [ ]:
#@title 1. Kết nối Google Drive, lấy/pull mã nguồn từ nhánh feat/m3-matching & chuẩn bị thư mục { display-mode: "form" }
import os, sys, subprocess
from pathlib import Path

BRANCH = "feat/m3-matching"
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    CODE_DIR = Path("/content/arxiv-code")
    DRIVE_DATA_DIR = Path("/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data")
    DRIVE_PARQUET_DIR = DRIVE_DATA_DIR / "parquet"
    DRIVE_CSV_DIR = DRIVE_DATA_DIR / "csv"

    if not (CODE_DIR / ".git").is_dir():
        print(f"--> Đang clone mã nguồn từ nhánh {BRANCH} về {CODE_DIR}...")
        subprocess.run([
            "git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
            "https://github.com/ahtuan171/arXiv-Project.git", str(CODE_DIR),
        ], check=True)
    else:
        print(f"--> Đang fetch & pull mã nguồn mới nhất từ nhánh {BRANCH}...")
        subprocess.run(["git", "-C", str(CODE_DIR), "fetch", "origin", BRANCH], check=True)
        actual = subprocess.check_output(
            ["git", "-C", str(CODE_DIR), "branch", "--show-current"], text=True
        ).strip()
        if actual != BRANCH:
            subprocess.run(["git", "-C", str(CODE_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "origin", BRANCH], check=True)

    LOCAL_PARQUET_DIR = Path("/content/local_parquet")
    LOCAL_OUT_PARQUET_DIR = Path("/content/local_out/parquet")
    LOCAL_OUT_CSV_DIR = Path("/content/local_out/csv")
else:
    CODE_DIR = Path.cwd().resolve()
    if not (CODE_DIR / "src").is_dir() and (CODE_DIR.parent / "src").is_dir():
        CODE_DIR = CODE_DIR.parent
    DRIVE_PARQUET_DIR = None
    DRIVE_CSV_DIR = None
    LOCAL_PARQUET_DIR = CODE_DIR / "data" / "parquet"
    LOCAL_OUT_PARQUET_DIR = LOCAL_PARQUET_DIR
    LOCAL_OUT_CSV_DIR = CODE_DIR / "data" / "csv"

os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("Môi trường        :", "Google Colab" if IN_COLAB else "Local")
print("Nhánh Git         :", BRANCH)
print("Mã nguồn          :", CODE_DIR)
print("Parquet làm việc  :", LOCAL_PARQUET_DIR)
if DRIVE_PARQUET_DIR:
    print("Parquet trên Drive:", DRIVE_PARQUET_DIR)


In [ ]:
#@title 2. Cài đặt thư viện (LightGBM, Scikit-learn, PyArrow) & Copy Parquet từ Drive về SSD cục bộ
import shutil
import subprocess, sys

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "scikit-learn", "pyarrow", "pandas", "psutil"], check=True)
    LOCAL_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_OUT_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_OUT_CSV_DIR.mkdir(parents=True, exist_ok=True)

    if not DRIVE_PARQUET_DIR.is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục Parquet trên Drive: {DRIVE_PARQUET_DIR}")

    for fname in ["papers.parquet", "references.parquet", "elements.parquet", "hierarchy.parquet", "bibtex.parquet"]:
        src = DRIVE_PARQUET_DIR / fname
        dst = LOCAL_PARQUET_DIR / fname
        if src.is_file():
            if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
                print(f"Đang copy {fname} ({src.stat().st_size / (1024*1024):.1f} MB) về SSD cục bộ...", flush=True)
                shutil.copy2(src, dst)
            else:
                print(f"Đã có sẵn trên SSD: {fname}")
        else:
            print(f"⚠️ Lưu ý: Không tìm thấy {src}")

    # Copy sẵn m3_manual_labels.csv nếu nhóm đã chuẩn bị trên Drive
    if DRIVE_CSV_DIR and (DRIVE_CSV_DIR / "m3_manual_labels.csv").is_file():
        shutil.copy2(DRIVE_CSV_DIR / "m3_manual_labels.csv", LOCAL_OUT_CSV_DIR / "m3_manual_labels.csv")
        print("Đã nạp m3_manual_labels.csv từ Drive.")
else:
    print("Chạy ở local, sử dụng trực tiếp file trong:", LOCAL_PARQUET_DIR)


In [ ]:
#@title 3. Cấu hình & Chạy Pipeline Milestone 3 (Huấn luyện 4 Mô hình + Benchmark Hệ thống) { display-mode: "form" }
#@markdown ### Giải thích các tham số cấu hình
#@markdown - **`AUTO_FRACTION`** *(tối thiểu `0.10`)*: Tỷ lệ phần trăm số bài báo ngoài tập gán nhãn thủ công được gán nhãn tự động bằng heuristic (mặc định `0.12` = 12%, đảm bảo vượt ngưỡng yêu cầu $\ge 10\%$ của đồ án). Tăng giá trị này giúp tập huấn luyện lớn hơn nhưng thời gian tạo cặp và train sẽ lâu hơn.
#@markdown - **`RANDOM_SEED`**: Hạt giống ngẫu nhiên dùng để cố định việc chọn bài báo gán nhãn, chia tập `train`/`valid`/`test` và khởi tạo các mô hình ML, giúp tái lập kết quả (reproducibility) giữa các lần chạy.
#@markdown - **`BUILD_HF_DATASET`**: Nếu bật (`True`), sau khi tạo `predictions.parquet`, pipeline sẽ tự động gom đủ 6 bảng (`papers`, `references`, `elements`, `hierarchy`, `bibtex`, `predictions`) kèm file `README.md` (YAML Dataset Card) vào thư mục `hf_dataset/` sẵn sàng upload lên Hugging Face.

AUTO_FRACTION = 0.12  #@param {type:"number"}
RANDOM_SEED = 42  #@param {type:"integer"}
BUILD_HF_DATASET = False  #@param {type:"boolean"}

from src.scripts.run_m3_pipeline import run_m3_pipeline

hf_dir = (LOCAL_OUT_PARQUET_DIR.parent / "hf_dataset") if BUILD_HF_DATASET else None
summary = run_m3_pipeline(
    parquet_dir=LOCAL_PARQUET_DIR,
    out_parquet_dir=LOCAL_OUT_PARQUET_DIR,
    csv_dir=LOCAL_OUT_CSV_DIR,
    auto_fraction=AUTO_FRACTION,
    seed=RANDOM_SEED,
    hf_dir=hf_dir,
)
print(f"\n✅ Hoàn tất Milestone 3 trong {summary['elapsed_sec']}s! Mô hình tốt nhất: {summary['model']['selected_model']}")


In [ ]:
#@title 4. Kết quả Benchmark 4 Mô hình, Tài nguyên Hệ thống & Feature Importance
import pandas as pd
from IPython.display import display

print("=== 1. THỐNG KÊ GÁN NHÃN (DATA LABELLING) ===")
display(pd.DataFrame([summary["labelling"]]))

print("\n=== 2. BẢNG SO SÁNH 4 MÔ HÌNH (MODEL & RANKING BENCHMARK) ===")
model_bench_df = pd.read_csv(LOCAL_OUT_CSV_DIR / "m3_model_benchmark.csv")
display(model_bench_df)
print("🏆 Mô hình được chọn xuất predictions.parquet:", summary["model"]["selected_model"])

print("\n=== 3. BENCHMARK TÀI NGUYÊN HỆ THỐNG (m3_system_benchmark.csv) ===")
sys_bench_df = pd.read_csv(LOCAL_OUT_CSV_DIR / "m3_system_benchmark.csv")
display(sys_bench_df)

print("\n=== 4. ĐỘ QUAN TRỌNG ĐẶC TRƯNG (PERMUTATION IMPORTANCE) ===")
feat_df = pd.read_csv(LOCAL_OUT_CSV_DIR / "m3_feature_importance.csv")
display(feat_df)


In [ ]:
#@title 5. Kiểm định toàn diện predictions.parquet (5 bước) & Đồng bộ lên Google Drive
import shutil
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from src.matching_module.exporter import PREDICTION_COLUMNS
from src.matching_module.model import compute_mrr_at_5

pred_path = LOCAL_OUT_PARQUET_DIR / "predictions.parquet"
df_pred = pd.read_parquet(pred_path)
df_bib_chk = pd.read_parquet(LOCAL_PARQUET_DIR / "bibtex.parquet", columns=["arxiv_id", "citation_key"])
df_ref_chk = pd.read_parquet(LOCAL_PARQUET_DIR / "references.parquet", columns=["arxiv_id", "ref_arxiv_id"])

print("=" * 70)
print("KIỂM ĐỊNH TOÀN DIỆN PREDICTIONS.PARQUET (5 BƯỚC)")
print("=" * 70)

# Bước 1: Schema & Non-null
pq_schema = pq.read_schema(pred_path)
assert list(df_pred.columns) == PREDICTION_COLUMNS, f"Sai cột: {list(df_pred.columns)}"
assert "dictionary" in str(pq_schema.field("partition").type), "partition phải là kiểu dictionary"
for req_col in ("arxiv_id", "partition", "citation_key"):
    assert int(df_pred[req_col].isna().sum()) == 0, f"Cột {req_col} có giá trị null!"
print("[1/5 PASS] Schema & Non-null đạt chuẩn 100%:", PREDICTION_COLUMNS)

# Bước 2: Độ đầy đủ 1-1 với bibtex.parquet
pred_papers = set(df_pred["arxiv_id"].astype(str))
bib_sub = df_bib_chk[df_bib_chk["arxiv_id"].astype(str).isin(pred_papers)]
bib_keys = set(zip(bib_sub["arxiv_id"].astype(str), bib_sub["citation_key"].astype(str)))
pred_keys = set(zip(df_pred["arxiv_id"].astype(str), df_pred["citation_key"].astype(str)))
missing_in_pred = len(bib_keys - pred_keys)
extra_in_pred = len(pred_keys - bib_keys)
dup_keys = int(df_pred.duplicated(subset=["arxiv_id", "citation_key"]).sum())
assert missing_in_pred == 0 and extra_in_pred == 0 and dup_keys == 0, (
    f"Lệch khóa bibtex: missing={missing_in_pred}, extra={extra_in_pred}, dup={dup_keys}"
)
print(f"[2/5 PASS] Đầy đủ 1-1 với bibtex.parquet: {len(df_pred):,} dòng | 0 thiếu | 0 dư | 0 trùng")

# Bước 3: Quy tắc chia tập & Chống rò rỉ dữ liệu (Anti-Leakage)
part_paper_counts = df_pred.groupby("partition", observed=True)["arxiv_id"].nunique().to_dict()
leakage_papers = int((df_pred.groupby("arxiv_id")["partition"].nunique() > 1).sum())
assert part_paper_counts.get("test") == 2 and part_paper_counts.get("valid") == 2, (
    f"Sai số bài test/valid: {part_paper_counts}"
)
assert leakage_papers == 0, f"Phát hiện {leakage_papers} bài xuất hiện ở nhiều hơn 1 partition!"
print(f"[3/5 PASS] Chia tập chuẩn & 0 data leakage | Số bài theo partition: {part_paper_counts}")

# Bước 4: Toàn vẹn tham chiếu với references.parquet & Không trùng ứng viên
ref_pairs = set(zip(df_ref_chk["arxiv_id"].astype(str), df_ref_chk["ref_arxiv_id"].astype(str)))
gt_nonnull = df_pred[df_pred["groundtruth_arxiv_id"].notna()]
gt_invalid = sum((str(r["arxiv_id"]), str(r["groundtruth_arxiv_id"])) not in ref_pairs for r in gt_nonnull.to_dict("records"))
cand_invalid = 0
cand_dups = 0
for r in df_pred.to_dict("records"):
    aid = str(r["arxiv_id"])
    cands = [str(r[f"cand_{i}"]) for i in range(1, 6) if pd.notna(r[f"cand_{i}"]) and str(r[f"cand_{i}"]).strip() != ""]
    if len(cands) != len(set(cands)):
        cand_dups += 1
    for c in cands:
        if (aid, c) not in ref_pairs:
            cand_invalid += 1
assert gt_invalid == 0 and cand_invalid == 0 and cand_dups == 0, (
    f"Lỗi tham chiếu: gt_invalid={gt_invalid}, cand_invalid={cand_invalid}, cand_dups={cand_dups}"
)
print("[4/5 PASS] Toàn vẹn tham chiếu 100% với references.parquet | 0 ứng viên trùng lặp")

# Bước 5: Chất lượng xếp hạng (MRR@5) & Xem mẫu trực quan (Spot-check)
test_metrics = compute_mrr_at_5(df_pred[df_pred["partition"] == "test"])
print(f"[5/5 PASS] Test MRR@5 kiểm chứng trực tiếp: {test_metrics}")
print("=" * 70)
print("🎉 KẾT LUẬN: predictions.parquet ĐẠT CHUẨN CHẤT LƯỢNG & ĐẦY ĐỦ 100%!")
display(df_pred[df_pred["partition"] == "test"].head(15))

if IN_COLAB and DRIVE_PARQUET_DIR and DRIVE_CSV_DIR:
    DRIVE_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_CSV_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pred_path, DRIVE_PARQUET_DIR / "predictions.parquet")
    print(f"\n✅ Đã xuất predictions.parquet lên Drive: {DRIVE_PARQUET_DIR / 'predictions.parquet'}")
    for fname in [
        "m3_model_benchmark.csv",
        "m3_system_benchmark.csv",
        "m3_feature_importance.csv",
        "m3_labelling_stats.csv",
        "m3_manual_labels.csv",
        "m3_evaluation_summary.json",
        "m3_best_model.pkl",
    ]:
        src_f = LOCAL_OUT_CSV_DIR / fname
        if src_f.is_file():
            shutil.copy2(src_f, DRIVE_CSV_DIR / fname)
    print(f"✅ Đã đồng bộ toàn bộ file benchmark & trọng số mô hình lên Drive: {DRIVE_CSV_DIR}")
